In [ ]:
%pip install -q pandas pyarrow boto3

In [ ]:
import gc
import io
import json
import re
from collections import defaultdict
from pathlib import Path

import boto3
import pandas as pd
from botocore.config import Config


# =============================================================================
# 可修改参数
# =============================================================================
MINIO_ENDPOINT = "https://iseadocker.isea.rwth-aachen.de:9000"
CREDENTIALS_FILE = Path("credentials.json")
BUCKET_NAME = "projects"
BASE_PREFIX = "j8005-metabatt/Metabatt/VTC/20_export_pulse/"

# None：遍历所有实际存在的电芯文件夹。
BATTERY_IDS = None
PROGRESS_EVERY = 10

CYCLE_ACTIVE_LIMIT_HOUR = 4.0
# 实验设置的 SOC 顺序；每个 Object_Key 内按 time diff 划分 cycle 后循环映射。
SOC_ORDER = ["90%", "50%", "10%"]
REQUIRED_SOC = {"10%", "50%", "90%"}
MIN_VALID_PULSE_COUNT = 1
MAX_VALID_PULSE_COUNT = 4

SOH_FILENAME_PATTERN = re.compile(
    r"BM\d+_(\d+(?:\.\d+)?)SOH\.parquet$",
    flags=re.IGNORECASE,
)
BATTERY_FOLDER_PATTERN = re.compile(
    r"^METABatt_Sony_Murata_18650VTC6_(\d+)/",
    flags=re.IGNORECASE,
)

if not CREDENTIALS_FILE.is_file():
    raise FileNotFoundError(
        f"找不到 {CREDENTIALS_FILE.resolve()}。请把 credentials.json 放在 notebook 工作目录中。"
    )

with CREDENTIALS_FILE.open(encoding="utf-8") as credential_file:
    credentials = json.load(credential_file)

try:
    MINIO_ACCESS_KEY = credentials["accessKey"].strip()
    MINIO_SECRET_KEY = credentials["secretKey"].strip()
except (KeyError, AttributeError) as exc:
    raise ValueError("credentials.json 必须包含 accessKey 和 secretKey 字符串。") from exc

if not MINIO_ACCESS_KEY or not MINIO_SECRET_KEY:
    raise ValueError("credentials.json 中的 Access Key 或 Secret Key 为空。")

print("已加载凭据：", MINIO_ACCESS_KEY[:4] + "..." + MINIO_ACCESS_KEY[-4:])

s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    region_name="us-east-1",
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"}),
)


In [ ]:
from botocore.exceptions import ClientError

try:
    result = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=BASE_PREFIX, MaxKeys=1)
    print("连接成功")
    print(result.get("Contents", []))
except ClientError as exc:
    print("错误详情：", exc.response.get("Error"))
    print("HTTP 状态：", exc.response.get("ResponseMetadata", {}).get("HTTPStatusCode"))
    raise


In [ ]:
def extract_soh_from_filename(filename):
    match = SOH_FILENAME_PATTERN.search(str(filename).strip())
    if match is None:
        raise ValueError(f"无法从文件名解析 SOH: {filename}")
    return float(match.group(1))


def list_parquet_objects_by_battery():
    """只保存对象键清单，不读取 parquet 原始数据。"""
    selected_ids = None if BATTERY_IDS is None else {int(x) for x in BATTERY_IDS}
    objects_by_battery = defaultdict(list)

    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=BUCKET_NAME, Prefix=BASE_PREFIX)
    for page in pages:
        for item in page.get("Contents", []):
            object_key = item["Key"]
            if not object_key.lower().endswith(".parquet"):
                continue

            relative_key = object_key[len(BASE_PREFIX):]
            match = BATTERY_FOLDER_PATTERN.match(relative_key)
            if match is None:
                continue

            battery_id = int(match.group(1))
            if selected_ids is not None and battery_id not in selected_ids:
                continue
            objects_by_battery[battery_id].append(object_key)

    if not objects_by_battery:
        raise FileNotFoundError(
            "未找到符合条件的 parquet 文件，请检查 BATTERY_IDS、BASE_PREFIX 和连接地址。"
        )

    return {
        battery_id: sorted(object_keys)
        for battery_id, object_keys in sorted(objects_by_battery.items())
    }


def read_one_battery(battery_id, object_keys):
    """只把一个电芯的数据读入内存；不保留文件级明细输出。"""
    frames = []
    skipped_reasons = []
    excluded_soh_conditions = []
    required_columns = {"Time", "Voltage", "Zustand"}

    for object_key in object_keys:
        filename = object_key.rsplit("/", 1)[-1]
        try:
            soh = extract_soh_from_filename(filename)
        except (TypeError, ValueError):
            excluded_soh_conditions.append({
                "Battery_ID": battery_id,
                "File": filename,
                "SOH": pd.NA,
                "Condition": "SOH 缺失",
            })
            continue

        if soh > 100:
            excluded_soh_conditions.append({
                "Battery_ID": battery_id,
                "File": filename,
                "SOH": soh,
                "Condition": "SOH > 100%",
            })
            continue

        try:
            response = s3.get_object(Bucket=BUCKET_NAME, Key=object_key)
            body = response["Body"]
            try:
                frame = pd.read_parquet(io.BytesIO(body.read()))
            finally:
                body.close()
        except Exception as exc:
            skipped_reasons.append(f"读取失败({type(exc).__name__})")
            continue

        missing_columns = required_columns.difference(frame.columns)
        if missing_columns:
            skipped_reasons.append("缺少必要列")
            continue

        frame["Battery_ID"] = battery_id
        frame["File"] = filename
        frame["Object_Key"] = object_key
        frame["SOH"] = soh
        frames.append(frame)

    if not frames:
        return pd.DataFrame(), skipped_reasons, excluded_soh_conditions
    return (
        pd.concat(frames, ignore_index=True),
        skipped_reasons,
        excluded_soh_conditions,
    )


In [ ]:
def check_soc_and_pulses(df):
    """按 time diff 划分 cycle/SOC，并返回严格三 SOC、每个 SOC 1–4 个脉冲检查表。"""
    data = df.copy()
    data["Time"] = pd.to_datetime(data["Time"], utc=True, errors="coerce")
    data["Voltage"] = pd.to_numeric(data["Voltage"], errors="coerce")
    data["Zustand"] = data["Zustand"].astype(str)

    key_columns = ["Time", "Voltage", "Battery_ID", "SOH", "Object_Key"]
    invalid_row_count = int(data[key_columns].isna().any(axis=1).sum())
    data = (
        data.dropna(subset=key_columns)
        .sort_values(["Object_Key", "Time"])
        .reset_index(drop=True)
    )

    empty_cycle_columns = [
        "Battery_ID", "Object_Key", "File", "SOH", "cycle_id",
        "SOC", "Pulse_Count", "Has_Usable_Pulses",
    ]
    empty_soh_columns = [
        "Battery_ID", "SOH", "Present_SOC", "Has_10_50_90_SOC",
        "All_SOC_Have_Usable_Pulses", "Passed",
    ]
    if data.empty:
        return (
            pd.DataFrame(columns=empty_cycle_columns),
            pd.DataFrame(columns=empty_soh_columns),
            invalid_row_count,
        )

    data["time_diff_hour"] = (
        data.groupby("Object_Key")["Time"].diff() / pd.Timedelta(hours=1)
    )
    data["is_new_cycle"] = (
        data["time_diff_hour"].isna()
        | data["time_diff_hour"].gt(CYCLE_ACTIVE_LIMIT_HOUR)
    )
    data["cycle_id"] = data.groupby("Object_Key")["is_new_cycle"].cumsum().astype(int)

    # cycle_id 从 1 开始，
    # 按实验顺序 90% -> 50% -> 10% 循环映射.
    data["SOC"] = data["cycle_id"].map(
        lambda cycle_id: SOC_ORDER[(cycle_id - 1) % len(SOC_ORDER)]
    )

    cycle_keys = ["Battery_ID", "Object_Key", "File", "SOH", "cycle_id"]
    cycle_soc = data[cycle_keys + ["SOC"]].drop_duplicates().reset_index(drop=True)

    # 先统一 CHA*/DCH*，再按连续 Zustand 划分脉冲段。
    data.loc[data["Zustand"].str.startswith("DCH", na=False), "Zustand"] = "DCH"
    data.loc[data["Zustand"].str.startswith("CHA", na=False), "Zustand"] = "CHA"
    data["is_pulse"] = data["Zustand"].isin(["CHA", "DCH"])
    data["pulse_segment_id"] = (
        data["Object_Key"].ne(data["Object_Key"].shift())
        | data["Zustand"].ne(data["Zustand"].shift())
    ).cumsum()

    pulse_count = (
        data.loc[data["is_pulse"]]
        .groupby(cycle_keys + ["SOC"], dropna=False)["pulse_segment_id"]
        .nunique()
        .rename("Pulse_Count")
        .reset_index()
    )
    cycle_check = cycle_soc.merge(pulse_count, on=cycle_keys + ["SOC"], how="left")
    cycle_check["Pulse_Count"] = cycle_check["Pulse_Count"].fillna(0).astype(int)
    cycle_check["Has_Usable_Pulses"] = (
        cycle_check["SOC"].isin(REQUIRED_SOC)
        & cycle_check["Pulse_Count"].between(
            MIN_VALID_PULSE_COUNT, MAX_VALID_PULSE_COUNT
        )
    )

    soh_rows = []
    for (battery_id, soh), group in cycle_check.groupby(["Battery_ID", "SOH"], sort=True):
        present_soc = set(group["SOC"].dropna())
        required_cycles = group[group["SOC"].isin(REQUIRED_SOC)]
        soh_rows.append({
            "Battery_ID": battery_id,
            "SOH": soh,
            "Present_SOC": ", ".join(
                soc for soc in ["10%", "50%", "90%"] if soc in present_soc
            ),
            "Has_10_50_90_SOC": REQUIRED_SOC.issubset(present_soc),
            "All_SOC_Have_Usable_Pulses": (
                not required_cycles.empty and required_cycles["Has_Usable_Pulses"].all()
            ),
        })

    soh_check = pd.DataFrame(soh_rows)
    if soh_check.empty:
        soh_check = pd.DataFrame(columns=empty_soh_columns)
    else:
        soh_check["Passed"] = (
            soh_check["Has_10_50_90_SOC"]
            & soh_check["All_SOC_Have_Usable_Pulses"]
        )
    return cycle_check, soh_check, invalid_row_count


In [ ]:
#   先单独剔除 SOH 缺失和 SOH > 100% 的 parquet，再按剩余 check-up / 三SOC 条件做电芯级分类。
def _format_numeric_range(series, digits=2):
    values = pd.to_numeric(series, errors="coerce").dropna()
    if values.empty:
        return "未知"
    low, high = values.min(), values.max()
    if abs(high - low) < 10 ** (-(digits + 1)):
        return f"{low:.{digits}f}"
    return f"{low:.{digits}f}–{high:.{digits}f}"


def _compress_id_ranges(values):
    """例如 [3,4,5,8,10,11] -> '003–005, 008, 010–011'。"""
    ids = sorted({int(value) for value in values if pd.notna(value)})
    if not ids:
        return "无"

    ranges = []
    start = previous = ids[0]
    for current in ids[1:]:
        if current == previous + 1:
            previous = current
            continue
        ranges.append((start, previous))
        start = previous = current
    ranges.append((start, previous))

    return ", ".join(
        f"{start:03d}" if start == end else f"{start:03d}–{end:03d}"
        for start, end in ranges
    )


def summarize_one_battery(
    battery_id, object_keys, data, soh_check, invalid_row_count, skipped_reasons,
    excluded_soh_conditions,
):
    file_count = len(object_keys)
    filtered_checkup_count = file_count - len(excluded_soh_conditions)
    readable_file_count = data["File"].nunique() if not data.empty else 0
    soh_values = (
        pd.to_numeric(data["SOH"], errors="coerce").dropna()
        if not data.empty else pd.Series(dtype="float64")
    )
    initial_soh = soh_values.max() if not soh_values.empty else pd.NA
    final_soh = soh_values.min() if not soh_values.empty else pd.NA
    coverage = (
        initial_soh - final_soh
        if pd.notna(initial_soh) and pd.notna(final_soh) else pd.NA
    )

    passed_soh_count = (
        int(soh_check.loc[soh_check["Passed"], "SOH"].nunique())
        if not soh_check.empty else 0
    )
    three_soc_soh_count = (
        int(soh_check.loc[soh_check["Has_10_50_90_SOC"], "SOH"].nunique())
        if not soh_check.empty and "Has_10_50_90_SOC" in soh_check.columns else 0
    )
    soh_stage_count = int(soh_values.nunique())
    has_three_soc_protocol = (
        soh_stage_count > 0
        and three_soc_soh_count == soh_stage_count
    )
    failed_soh_count = max(soh_stage_count - passed_soh_count, 0)

    abnormal_parts = []
    if skipped_reasons:
        abnormal_parts.append(f"无法使用的文件 {len(skipped_reasons)} 个")
    if invalid_row_count:
        abnormal_parts.append(f"关键字段无效行 {invalid_row_count} 行")
    if failed_soh_count:
        abnormal_parts.append(
            f"未通过严格三 SOC/每个 SOC 1–4 个脉冲筛选的 SOH 阶段 {failed_soh_count} 个"
        )

    exclude_reasons = []
    if filtered_checkup_count <= 3:
        exclude_reasons.append("check-up ≤ 3")
    if not has_three_soc_protocol:
        exclude_reasons.append("不满足10%、50%、90%三SOC协议")

    # A完整的具体条件：未触发电芯级剔除，且通过 SOH 点数不少于 5。
    a_condition_no_exclusion = not exclude_reasons
    a_condition_soh_count = passed_soh_count >= 5
    a_condition_passed = (
        a_condition_no_exclusion
        and a_condition_soh_count
    )
    b_condition_reasons = []
    if not a_condition_soh_count:
        b_condition_reasons.append("合格 SOH 数量 < 5")
    b_condition_reason = "；".join(b_condition_reasons) if b_condition_reasons else "无"

    if exclude_reasons:
        screening_class = "剔除"
        usage = "不用"
    elif a_condition_passed:
        screening_class = "A完整"
        usage = "train/val/test"
    else:
        screening_class = "B受限"
        usage = "仅train"

    if not data.empty:
        temperature = (
            _format_numeric_range(data["Temperature"], 2)
            if "Temperature" in data.columns else "未知"
        )
    else:
        temperature = "未知"
        abnormal_parts.append("该文件夹没有可用数据")

    return {
        "电芯编号": battery_id,
        "电芯文件夹": f"METABatt_Sony_Murata_18650VTC6_{battery_id:03d}",
        "文件数量": file_count,
        "单独剔除后 check-up 数量": filtered_checkup_count,
        "成功读取文件数": int(readable_file_count),
        "通过筛选的 SOH 数量": passed_soh_count,
        "满足三SOC的 SOH 数量": three_soc_soh_count,
        "是否满足三SOC协议": has_three_soc_protocol,
        "初始 SOH": initial_soh,
        "最终 SOH": final_soh,
        "SOH 覆盖范围": (
            f"{final_soh:.1f}–{initial_soh:.1f}"
            if pd.notna(initial_soh) and pd.notna(final_soh) else "未知"
        ),
        "SOH 覆盖跨度": coverage,
        "温度": temperature,
        "筛选分类": screening_class,
        "A条件_电芯级未剔除": a_condition_no_exclusion,
        "A条件_通过SOH数量≥5": a_condition_soh_count,
        "A条件结果": "通过" if a_condition_passed else "未通过",
        "B条件结果": "通过" if screening_class == "B受限" else "未通过",
        "B具体未满足条件": b_condition_reason if screening_class == "B受限" else "无",
        "剔除原因": "；".join(exclude_reasons) if exclude_reasons else "无",
        "用途": usage,
        "是否存在缺失或异常": bool(abnormal_parts),
        "异常说明": "；".join(abnormal_parts) if abnormal_parts else "无",
    }


objects_by_battery = list_parquet_objects_by_battery()
total_batteries = len(objects_by_battery)
print(f"共发现 {total_batteries} 个实际存在的电芯文件夹。")

battery_rows = []
condition_records = []
for position, (battery_id, object_keys) in enumerate(objects_by_battery.items(), start=1):
    battery_data, skipped_reasons, excluded_soh_conditions = read_one_battery(
        battery_id, object_keys
    )
    condition_records.extend(excluded_soh_conditions)
    if battery_data.empty:
        cycle_check = pd.DataFrame()
        soh_check = pd.DataFrame(columns=["SOH", "Passed"])
        invalid_row_count = 0
    else:
        cycle_check, soh_check, invalid_row_count = check_soc_and_pulses(battery_data)

    battery_rows.append(summarize_one_battery(
        battery_id=battery_id,
        object_keys=object_keys,
        data=battery_data,
        soh_check=soh_check,
        invalid_row_count=invalid_row_count,
        skipped_reasons=skipped_reasons,
        excluded_soh_conditions=excluded_soh_conditions,
    ))

    # 不保留该电芯的原始数据和文件级检查明细。
    del battery_data, cycle_check, soh_check
    gc.collect()

    if position == 1 or position % PROGRESS_EVERY == 0 or position == total_batteries:
        print(f"处理进度：{position}/{total_batteries}，当前电芯 {battery_id:03d}")

battery_summary = (
    pd.DataFrame(battery_rows)
    .sort_values("电芯编号")
    .reset_index(drop=True)
)
for _, row in battery_summary.iterrows():
    if row["剔除原因"] == "无":
        continue
    for condition in row["剔除原因"].split("；"):
        condition_records.append({
            "Battery_ID": row["电芯编号"],
            "File": pd.NA,
            "SOH": pd.NA,
            "Condition": condition,
        })

condition_details = pd.DataFrame(condition_records)
condition_summary = (
    condition_details.groupby("Condition")
    .agg(
        筛出记录数量=("Battery_ID", "size"),
        电芯数量=("Battery_ID", "nunique"),
        电芯文件夹编号范围=("Battery_ID", _compress_id_ranges),
    )
    .reset_index()
    .rename(columns={"Condition": "筛选 condition"})
    if not condition_details.empty else pd.DataFrame(
        columns=["筛选 condition", "筛出记录数量", "电芯数量", "电芯文件夹编号范围"]
    )
)

soh_bins = [-float("inf"), 60, 70, 75, 80, 85, 90, 95, float("inf")]
soh_labels = ["≤60", "60–70", "70–75", "75–80", "80–85", "85–90", "90–95", ">95"]
final_soh_numeric = pd.to_numeric(battery_summary["最终 SOH"], errors="coerce")
battery_summary["最终 SOH 分组"] = pd.cut(
    final_soh_numeric, bins=soh_bins, labels=soh_labels, right=True
)
soh_groups = (
    battery_summary.groupby(
        ["最终 SOH 分组"],
        dropna=False,
        observed=False,
    )
    .agg(
        电芯数量=("电芯编号", "nunique"),
        电芯文件夹编号范围=("电芯编号", _compress_id_ranges),
    )
    .reset_index()
)
soh_groups["电芯文件夹编号范围"] = (
    soh_groups["电芯文件夹编号范围"].fillna("无")
)

quality_order = ["剔除", "A完整", "B受限"]
battery_summary["筛选分类"] = pd.Categorical(
    battery_summary["筛选分类"], categories=quality_order, ordered=True
)
quality_summary = (
    battery_summary.groupby("筛选分类", observed=False)
    .agg(
        电芯数量=("电芯编号", "nunique"),
        电芯文件夹编号范围=("电芯编号", _compress_id_ranges),
    )
    .reset_index()
)
quality_summary["用途"] = quality_summary["筛选分类"].map({
    "剔除": "不用",
    "A完整": "train/val/test",
    "B受限": "仅train",
})
non_excluded = battery_summary["筛选分类"].isin(["A完整", "B受限"])
a_complete = battery_summary["筛选分类"].eq("A完整")
b_limited = battery_summary["筛选分类"].eq("B受限")
condition_masks = [
    ("A条件1：电芯未触发剔除", "单独剔除异常 SOH parquet 后，check-up > 3，且剩余 SOH 阶段全部满足三SOC协议", non_excluded),
    ("A条件2：合格 SOH 数量 ≥ 5", "通过三 SOC 且每个 SOC 1–4 脉冲的 SOH 数量 ≥ 5", non_excluded & battery_summary["A条件_通过SOH数量≥5"]),
    ("A完整：条件1+条件2", "以上两个 A 条件同时满足", a_complete),
    ("B条件1：电芯未触发剔除", "单独剔除异常 SOH parquet 后，check-up > 3，且剩余 SOH 阶段全部满足三SOC协议", b_limited),
    ("B条件2：合格 SOH 数量 < 5", "合格 SOH 数量不足 5 个", b_limited & ~battery_summary["A条件_通过SOH数量≥5"]),
    ("B受限：条件1且条件2", "未触发剔除，且 A 条件2不满足", b_limited),
]
quality_condition_summary = pd.DataFrame([
    {"条件类别": label, "具体条件": desc, "满足电芯数量": int(mask.sum()),
     "满足电芯文件夹编号范围": _compress_id_ranges(battery_summary.loc[mask, "电芯编号"]) }
    for label, desc, mask in condition_masks
])
ab_condition_details = battery_summary.loc[
    battery_summary["筛选分类"].isin(["A完整", "B受限"]),
    [
        "电芯编号", "电芯文件夹", "筛选分类", "通过筛选的 SOH 数量",
        "A条件_电芯级未剔除", "A条件_通过SOH数量≥5",
        "A条件结果", "B条件结果", "B具体未满足条件", "用途",
    ],
].reset_index(drop=True)
excluded_batteries = battery_summary.loc[
    battery_summary["筛选分类"].eq("剔除"),
    [
        "电芯编号", "电芯文件夹", "文件数量", "成功读取文件数",
        "初始 SOH", "满足三SOC的 SOH 数量", "是否满足三SOC协议",
        "筛选分类", "剔除原因", "用途",
    ],
].reset_index(drop=True)

print("每个电芯文件夹摘要：")
display(battery_summary)
print("按最终 SOH 分组（仅记录电芯文件夹范围）：")
display(soh_groups)
print("电芯整体质量分类：")
display(quality_summary)
print("A/B 具体筛选条件汇总：")
display(quality_condition_summary)
print("A/B 电芯明细及逐项条件：")
display(ab_condition_details)
print("被剔除出深度学习数据集的电芯：")
display(excluded_batteries)
print("各筛选 condition 对应的电芯：")
display(condition_summary)

# battery_summary.to_csv("battery_summary.csv", index=False, encoding="utf-8-sig")
# soh_groups.to_csv("soh_groups.csv", index=False, encoding="utf-8-sig")
